# 🤖 Qwen2.5-7B RAG API Server on Kaggle

This notebook deploys a **LitServe API Server** running the Qwen2.5-7B-Instruct (GGUF) model on a Kaggle GPU and exposes it to the internet via an SSH tunnel.

## Workflow
| Step | Cell | Description |
|------|------|-------------|
| 1 | Cell 2 | Install dependencies |
| 2 | Cell 3 | Locate model file & check GPU |
| 3 | Cell 4 | Write server script to disk |
| 4 | Cell 5 | Start SSH tunnel → get Public URL |
| 5 | Cell 6 | Start LitServer **(blocking)** |
| 6 | Cell 7 | Send a test request from local machine |
##

> **Note:** Cell 6 runs indefinitely — this is expected. Use the URL from Cell 5 to call the API from your local machine.


In [ ]:
import subprocess, sys

# --- Step 1: Install llama-cpp-python with CUDA 12.4 support ---
print("Installing llama-cpp-python...")
r1 = subprocess.run(
    [sys.executable, "-m", "pip", "install", "llama-cpp-python",
     "--upgrade", "--force-reinstall", "--no-cache-dir",
     "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu124"],
    capture_output=True, text=True
)
print(r1.stdout[-2000:] if len(r1.stdout) > 2000 else r1.stdout)
if r1.returncode != 0:
    print("STDERR:", r1.stderr[-1000:])

# --- Step 2: Install LitServe (pinned to 0.2.7 to avoid MCPServer bug in newer versions) ---
print("\nInstalling LitServe 0.2.7...")
r2 = subprocess.run(
    [sys.executable, "-m", "pip", "install", "litserve==0.2.7", "-q"],
    capture_output=True, text=True
)
print("LitServe 0.2.7 OK" if r2.returncode == 0 else r2.stderr[-500:])


In [ ]:
import os, subprocess

# Default path on Kaggle — auto-searches /kaggle/input if not found
model_path = "/kaggle/input/qwen2-5-7b-instruct-gguf/Qwen2.5-7B-Instruct-Q4_K_M.gguf"

if not os.path.exists(model_path):
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".gguf"):
                model_path = os.path.join(root, f)
                break

print(f"Model path : {model_path}")
print(f"Exists     : {os.path.exists(model_path)}")

# Check GPU info
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True
)
print(f"\nGPU:\n{gpu.stdout.strip()}" if gpu.returncode == 0 else "No GPU found")


In [ ]:
# Write the server class to a separate .py file to avoid multiprocessing pickle errors in Jupyter.
# LitServer spawns worker processes — the class must be importable from a real module, not a notebook.

api_code = '''
import os
import threading
import litserve as ls
from llama_cpp import Llama

MODEL_PATH = "{model_path}"

# Bearer token for endpoint authentication.
# Set the LITSERVE_API_KEY env var on Kaggle (Add-ons → Secrets) before running.
API_KEY = os.environ.get("LITSERVE_API_KEY", "")
if not API_KEY:
    raise RuntimeError("LITSERVE_API_KEY env var is not set. Set it in Kaggle Secrets.")


class QwenRAGAPI(ls.LitAPI):
    def setup(self, device):
        self._lock = threading.Lock()
        self.llm = Llama(
            model_path=MODEL_PATH,
            n_gpu_layers=-1,   # offload entire model to GPU
            n_ctx=4096,        # context window: ~3000 usable tokens for RAG
            n_batch=256,
            verbose=False,
        )
        print("Model loaded!")

    def decode_request(self, request):
        # Expects: {{"messages": [...], "max_tokens": int, "temperature": float}}
        # Validate bearer token from Authorization header.
        auth_header = request.headers.get("authorization", "")
        if not auth_header.startswith("Bearer ") or auth_header[7:] != API_KEY:
            raise ls.HTTPException(status_code=401, detail="Unauthorized")
        return request.json()

    def predict(self, request):
        messages    = request.get("messages", [])
        max_tokens  = request.get("max_tokens", 512)
        temperature = request.get("temperature", 0.7)
        # llama-cpp is not thread-safe — serialize with a lock
        with self._lock:
            result = self.llm.create_chat_completion(
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature,
                top_p=0.9,
                repeat_penalty=1.1,
            )
        return result["choices"][0]["message"]["content"]

    def encode_response(self, output):
        return {{"response": output}}

if __name__ == "__main__":
    server = ls.LitServer(
        QwenRAGAPI(),
        workers_per_device=1,  # required: llama-cpp does not support multi-threading
        timeout=120,
    )
    print("LitServer starting on port 8000...")
    server.run(port=8000)
'''

import os
with open("/kaggle/working/qwen_server.py", "w") as f:
    f.write(api_code.format(model_path=model_path))

print("✓ Written to /kaggle/working/qwen_server.py")
print(f"  Model: {model_path}")


In [ ]:
import subprocess, threading, re, time

tunnel_url = {"value": None}

# ──────────────────────────────────────────────────────────
# Select tunnel method:
#   "localhost.run" — SSH tunnel, no account required (recommended)
#   "pinggy"        — SSH over port 443, use if localhost.run is blocked
#   "ngrok"         — Requires a free account at https://ngrok.com
# ──────────────────────────────────────────────────────────
TUNNEL_MODE = "localhost.run"
NGROK_TOKEN = ""  # Only needed if TUNNEL_MODE = "ngrok"

def run_tunnel():
    if TUNNEL_MODE == "localhost.run":
        cmd = [
            "ssh", "-o", "StrictHostKeyChecking=no",
            "-o", "ServerAliveInterval=30",
            "-R", "80:localhost:8000",
            "nokey@localhost.run"
        ]
        pattern = r"https://[a-zA-Z0-9\-]+\.localhost\.run"

    elif TUNNEL_MODE == "pinggy":
        cmd = [
            "ssh", "-p", "443",
            "-o", "StrictHostKeyChecking=no",
            "-o", "ServerAliveInterval=30",
            "-R", "0:localhost:8000",
            "a.pinggy.io"
        ]
        pattern = r"https://[a-zA-Z0-9\-]+\.a\.pinggy\.io"

    elif TUNNEL_MODE == "ngrok":
        if not NGROK_TOKEN:
            raise ValueError("NGROK_TOKEN must be set before using ngrok!")
        subprocess.run(["pip", "install", "pyngrok", "-q"], capture_output=True)
        from pyngrok import ngrok, conf
        conf.get_default().auth_token = NGROK_TOKEN
        public_url = ngrok.connect(8000).public_url
        tunnel_url["value"] = public_url
        print(f"\n{'='*55}\n  PUBLIC API URL: {public_url}\n{'='*55}\n")
        return  # ngrok manages its own connection loop

    else:
        raise ValueError(f"Invalid TUNNEL_MODE: {TUNNEL_MODE}")

    # Run in a separate session so it is not killed by SIGINT when the cell finishes
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        start_new_session=True,
    )
    for line in proc.stdout:
        print(line, end="")
        match = re.search(pattern, line)
        if match:
            tunnel_url["value"] = match.group(0)
            print(f"\n{'='*55}\n  PUBLIC API URL: {tunnel_url['value']}\n{'='*55}\n")

t = threading.Thread(target=run_tunnel, daemon=True)
t.start()
print(f"[{TUNNEL_MODE}] Tunnel connecting... Run the next cell to start the server.")


In [ ]:
# ⚠️  This cell blocks indefinitely — that is expected behavior.
# The server is running and listening for requests. Do NOT stop this cell.
# The public URL was printed in Cell 4.

import subprocess, sys

proc = subprocess.Popen(
    [sys.executable, "/kaggle/working/qwen_server.py"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

# Stream server logs to the notebook output
for line in proc.stdout:
    print(line, end="", flush=True)


In [ ]:
# Run this cell from your LOCAL machine (not Kaggle) to test the API.
# Replace API_URL and API_KEY with the values from Cell 5 and your Kaggle Secret.
import requests

API_URL = "your_url"  # ← replace with URL from Cell 5
API_KEY = "your_key"  # ← replace with your LITSERVE_API_KEY value

# Example: inject retrieved context into the system message (RAG pattern)
context = """[Reference Document]
Title: NovaCore System Technical Documentation v4.2
...
"""

response = requests.post(
    f"{API_URL}/predict",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "messages": [
            {"role": "system", "content": f"Answer based on the following document:\n{context}"},
            {"role": "user",   "content": "What is this document about?"}
        ],
        "max_tokens": 512,
        "temperature": 0.7,
    },
    timeout=120,
)
print(response.json()["response"])


Tài liệu này mô tả về kiến trúc mạng thần kinh mới được phát triển bởi tập đoàn Aetheria Labs mang tên NovaCore v4.2. Nó giới thiệu tổng quan về công nghệ, thông số kỹ thuật và hiệu năng của NovaCore, cũng như cơ sở hạ tầng và môi trường huấn luyện. Tài liệu này còn bao gồm lịch sử cập nhật và xử lý sự cố trong quá trình phát triển của NovaCore.

Một điểm nổi bật chính là việc NovaCore loại bỏ cơ chế Self-Attention tiêu chuẩn để chuyển sang giao thức "Dynamic Synapse Routing" (DSR), cho phép mô hình tự động điều chỉnh trọng số kết nối trong thời gian thực dựa trên độ phức tạp của câu hỏi đầu vào.
